In [1]:
import pandas as pd

df = pd.read_csv("data/raw/bigmart_sales.csv")
print(df.shape)
df.head()


(8523, 12)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [2]:
df.shape

(8523, 12)

In [3]:
df.dtypes

Item_Identifier                  str
Item_Weight                  float64
Item_Fat_Content                 str
Item_Visibility              float64
Item_Type                        str
Item_MRP                     float64
Outlet_Identifier                str
Outlet_Establishment_Year      int64
Outlet_Size                      str
Outlet_Location_Type             str
Outlet_Type                      str
Item_Outlet_Sales            float64
dtype: object

In [4]:
df.isnull().sum()

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df["Item_Fat_Content"].value_counts()

Item_Fat_Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: int64

In [7]:
df["Outlet_Size"].value_counts(dropna=False)

Outlet_Size
Medium    2793
NaN       2410
Small     2388
High       932
Name: count, dtype: int64

In [8]:
df["Outlet_Type"].value_counts()

Outlet_Type
Supermarket Type1    5577
Grocery Store        1083
Supermarket Type3     935
Supermarket Type2     928
Name: count, dtype: int64

In [9]:
(df["Item_Visibility"] == 0).sum()

np.int64(526)

In [10]:
df.duplicated(subset=["Item_Identifier", "Outlet_Identifier"]).sum()

np.int64(0)

In [11]:
df[df["Outlet_Size"].isnull()]["Outlet_Identifier"].value_counts()

Outlet_Identifier
OUT045    929
OUT017    926
OUT010    555
Name: count, dtype: int64

In [12]:
df_clean = df.copy()

fat_map = {"LF" : "Low Fat", "low fat": "Low Fat", "reg": "Regular"}
df_clean["Item_Fat_Content"] = df_clean["Item_Fat_Content"].replace(fat_map)

df_clean["Item_Fat_Content"].value_counts()

Item_Fat_Content
Low Fat    5517
Regular    3006
Name: count, dtype: int64

In [13]:
df_clean["Item_Weight"] = df_clean["Item_Weight"].fillna(df_clean["Item_Weight"].median())

In [14]:
df_clean["Outlet_Size"] = df_clean["Outlet_Size"].fillna("Unknown")

In [15]:
df_clrean = df_clean.drop_duplicates()


In [16]:
print(df_clean.isnull().sum())

print(df_clean["Item_Fat_Content"].value_counts())

print(df_clean.shape)

Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64
Item_Fat_Content
Low Fat    5517
Regular    3006
Name: count, dtype: int64
(8523, 12)


In [17]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

load_dotenv()

url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(url)

df_clean.to_sql("bigmart_sales", engine, if_exists="replace", index=False)
print("Loaded", len(df_clean), "rows") 

Loaded 8523 rows


In [18]:
df_clean["Item_Outlet_Sales"].sum()

np.float64(18591125.4104)